# IFC element exploration

Explore and extract objects from an IFC file.


In [1]:
from __future__ import annotations

import re
from collections import Counter
from pathlib import Path

import ifcopenshell
import pandas as pd

# Path to the IFC file (default assumes notebook sits in project root/model_explore)
IFC_PATH = Path("./HSK_HSK1A_GVDC_ALL_IFC4x3.ifc").resolve()
IFC_PATH

WindowsPath('C:/Users/benchen/code/python/model_explore/HSK_HSK1A_GVDC_ALL_IFC4x3.ifc')

In [2]:
# Quick check the IFC file exists and report size
if not IFC_PATH.exists():
    raise FileNotFoundError(f"IFC file not found at {IFC_PATH}")

size_mb = IFC_PATH.stat().st_size / (1024 * 1024)
print(f"Using IFC file: {IFC_PATH}")
print(f"Size: {size_mb:.1f} MB")

Using IFC file: C:\Users\benchen\code\python\model_explore\HSK_HSK1A_GVDC_ALL_IFC4x3.ifc
Size: 414.3 MB


In [3]:
# Load the IFC model
ifc_file = ifcopenshell.open(str(IFC_PATH))

In [4]:
print(f"Loaded schema: {ifc_file.schema}")

Loaded schema: IFC4X3


In [5]:
# Count entities by IFC type
counts = Counter(ent.is_a() for ent in ifc_file)

df_counts = (
    pd.DataFrame(counts.items(), columns=["ifc_type", "count"])
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)
df_counts.head(25)

,ifc_type,count
0,IfcPolyLoop,2248622
1,IfcFaceOuterBound,2244388
2,IfcFace,2244388
3,IfcCartesianPoint,1444132
4,IfcStyledItem,7571
5,IfcClosedShell,6818
6,IfcFacetedBrep,6818
7,IfcPropertySet,5503
8,IfcRelDefinesByProperties,5209
9,IfcAxis2Placement3D,5049


In [6]:
from collections import Counter

types = [instance.is_a() for instance in ifc_file.by_type("IfcElement")] + \
        [instance.is_a() for instance in ifc_file.by_type("IfcProduct")]  # catches some extra

counts = Counter(types)

# Print sorted by count
for ifc_type, count in counts.most_common():
    print(f"{ifc_type:30} {count}")

print(f"total elements: {sum(counts.values())}")

IfcBuildingElementProxy        2050
IfcOpeningElement              1028
IfcBeam                        574
IfcSlab                        338
IfcWall                        240
IfcDoor                        158
IfcFurniture                   120
IfcBuildingStorey              40
IfcCovering                    38
IfcAnnotation                  3
IfcBuilding                    1
IfcSite                        1
total elements: 4591


In [7]:
# filter element by name (contains a certain string)

keyword = "MiC"
blacklist = ["DOWEL", "Slab Edge", "JOIN"]

mic_elements = []
mic_types = []

cnt = 0
for elem in ifc_file.by_type("IfcElement"):
    if elem.is_a("IfcBuildingElementProxy") and \
        "MiC" in elem.Name and all([b not in elem.Name for b in blacklist]):
        cnt += 1
        print(elem.Name)
        mic_elements.append(elem)
        family_name, type_name, elem_id = elem.Name.split(":")
        mic_types.append(":".join([family_name, type_name]))

mic_types_counter = Counter(mic_types)

print(f"total count: {cnt}\ntotal types: {len(set(mic_types))}")
for k, v in mic_types_counter.items():
    print(f"{k}: {v}")

MiC_TYPE 3.2_LK2L:MiC_TYPE 3.2_LK2L:57129635
MiC_TYPE 3.1_LK1L:MiC_TYPE 3.1_LK1L:57129862
MiC_TYPE 3_LK1L:MiC_TYPE 3_LK1L:57130029
MiC_TYPE 3B1_LD2L:MiC_TYPE 3B1_LD2L:57130471
MiC_TYPE 3B2_LD1L & LD2R:MiC_TYPE 3B2_LD1L & LD2R:57130561
MiC_TYPE 3B2_LD1L & LD2R:MiC_TYPE 3B2_LD1L & LD2R:57130577
MiC_TYPE 1.2_MB2L & MB2R:MiC_TYPE 1.2_MB2L & MB2R:57130784
MiC_TYPE 2_BT3L & BT3R:MiC_TYPE 2_BT3L & BT3R:57131005
MiC_TYPE 1.1_MB3R:MiC_TYPE 1.1_MB3R:57131994
MiC_TYPE 5.1_BR2:MiC_TYPE 5.1_BR2:57132416
MiC_TYPE 6_TL:MiC_TYPE 6_TL:57133352
MiC_TYPE 1_MB1R:MiC_TYPE 1_MB1R:57133768
MiC_TYPE 5_BR1:MiC_TYPE 5_BR1:57133952
MiC_TYPE 6_TL:MiC_TYPE 6_TL:57160681
MiC_TYPE 1.2_MB3:MiC_TYPE 1.2_MB3:57161344
MiC_TYPE 2.1_BT4:MiC_TYPE 2.1_BT4:57185519
MiC_TYPE 2.2_BT3R:MiC_TYPE 2.2_BT3R:57185898
MiC_TYPE 4.1_KTL & KTR:4.1:57186426
MiC_TYPE 4.1_KTL & KTR:4.1R:57186783
MiC_TYPE 2B-P3_BT1L:MiC_TYPE 2B-P3_BT1L:57187295
MiC_TPYE 1.3_MB1L & MB1R:MiC_TPYE 1.3_MB1L & MB1R:57187604
MiC_TPYE 1.3_MB1L & MB1R:MiC_TPYE 1.3_

In [16]:
import re
import csv
import time
import logging
from pathlib import Path

import ifcopenshell


def export_elements_to_individual_ifc_ifcpatch(
    elements,
    output_dir,
    cur_file,
    csv_filename="export_index.csv",
    log_every=1,
    logger=None,
):
    """FAST: Export each element to its own IFC using IfcPatch ExtractElements.

    Why this is fast for huge IFCs:
      - It builds a NEW model containing only the extracted element(s)
      - Avoids deleting millions of entities (slow)

    How the query works:
      - ExtractElements uses ifcopenshell.util.selector.filter_elements() "filter elements syntax".
      - To select ONE object, pass ONLY its GlobalId string (no brackets, no dots).
        Example: "3OJkD8bnr7ZBGamr9ZFQ2E".  

    Outputs:
      - One IFC file per element
      - A CSV mapping filename -> OriginalName and OriginalName split by ':' (Part1..PartN)
    """
    try:
        import ifcpatch
    except ImportError as e:
        raise ImportError(
            "ifcpatch is not available. Update IfcOpenShell (recommended) or install ifcpatch."
        ) from e

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Logger setup
    if logger is None:
        logger = logging.getLogger("ifc_export")
        if not logger.handlers:
            h = logging.StreamHandler()
            h.setFormatter(logging.Formatter("[%(asctime)s] %(levelname)s: %(message)s"))
            logger.addHandler(h)
        logger.setLevel(logging.INFO)

    elements = list(elements)
    total = len(elements)
    logger.info(f"Starting export (ifcpatch): {total} element(s) -> {output_dir}")

    rows = []

    for i, el in enumerate(elements, start=1):
        gid = getattr(el, "GlobalId", None)
        orig_name = getattr(el, "Name", None) or ""
        ifc_class = el.is_a() if hasattr(el, "is_a") else "(unknown)"

        if (i % log_every) == 0 or i == 1 or i == total:
            logger.info(f"[{i}/{total}] Extracting {ifc_class} | GlobalId={gid} | Name='{orig_name}'")

        if not gid:
            logger.warning("  - missing GlobalId; skipping")
            continue

        # ✅ Correct query syntax for ExtractElements / filter_elements:
        # Just the GlobalId string.
        query = str(gid).strip()

        t_patch = time.time()
        out = ifcpatch.execute({
            "file": cur_file,
            "recipe": "ExtractElements",
            "arguments": [query],
        })
        logger.info(f"  - ifcpatch ExtractElements done in {time.time() - t_patch:.2f}s")

        # Filename
        base_name = orig_name or ifc_class
        safe_name = re.sub(r"[^0-9A-Za-z_\-]+", "_", str(base_name)).strip("_") or ifc_class
        filename = f"{safe_name}_{query}.ifc"
        filepath = output_dir / filename

        # Ensure uniqueness
        if filepath.exists():
            k = 2
            while True:
                filename2 = f"{safe_name}_{query}_{k}.ifc"
                filepath2 = output_dir / filename2
                if not filepath2.exists():
                    filename, filepath = filename2, filepath2
                    break
                k += 1

        # Write IFC
        t_wr = time.time()
        if hasattr(out, "write"):
            out.write(str(filepath))
        elif hasattr(ifcpatch, "write"):
            ifcpatch.write(out, str(filepath))
        else:
            raise RuntimeError("IfcPatch returned an unsupported output object; cannot write IFC")

        size_mb = filepath.stat().st_size / (1024 * 1024)
        logger.info(f"  - wrote {filename} in {time.time() - t_wr:.2f}s (size={size_mb:.2f} MB)")

        # CSV row with Name split by ':'
        parts = [p.strip() for p in orig_name.split(":")] if orig_name else []
        rows.append({
            "filename": filename,
            "GlobalId": query,
            "IfcClass": ifc_class,
            "OriginalName": orig_name,
            "parts": parts,
        })

    # Write CSV index
    if rows:
        max_parts = max((len(r["parts"]) for r in rows), default=0)
        headers = ["filename", "GlobalId", "IfcClass", "OriginalName"] + [f"Part{i}" for i in range(1, max_parts + 1)]

        csv_path = output_dir / csv_filename
        with open(csv_path, "w", newline="", encoding="utf-8-sig") as f:
            w = csv.DictWriter(f, fieldnames=headers)
            w.writeheader()
            for r in rows:
                out_row = {
                    "filename": r["filename"],
                    "GlobalId": r["GlobalId"],
                    "IfcClass": r["IfcClass"],
                    "OriginalName": r["OriginalName"],
                }
                for idx in range(max_parts):
                    out_row[f"Part{idx+1}"] = r["parts"][idx] if idx < len(r["parts"]) else ""
                w.writerow(out_row)

        logger.info(f"CSV index written: {csv_path}")
        return str(csv_path)

    logger.warning("No exports succeeded; CSV index not written.")
    return None


# -------------------------
# Example usage
# -------------------------
#
# from ifcopenshell.util.selector import Selector
# ifc_file = ifcopenshell.open("big_model.ifc")
# selector = Selector()
# mic_elements = selector.parse(ifc_file, '.IfcBuildingElementProxy[Name~/MiC_TYPE/]' )
# csv_path = export_elements_to_individual_ifc_ifcpatch(mic_elements, "mic_individuals", ifc_file)
# print("CSV:", csv_path)


In [17]:
output_folder = Path("mic_individuals")
output_folder.mkdir(exist_ok=True)

export_elements_to_individual_ifc_ifcpatch(mic_elements, output_folder, ifc_file)

[2025-12-12 15:34:31,905] INFO: Starting export (ifcpatch): 49 element(s) -> mic_individuals
[2025-12-12 15:34:31,906] INFO: [1/49] Extracting IfcBuildingElementProxy | GlobalId=3OJkD8bnr7ZBGamr9ZFQ2E | Name='MiC_TYPE 3.2_LK2L:MiC_TYPE 3.2_LK2L:57129635'
[2025-12-12 15:35:04,580] INFO:   - ifcpatch ExtractElements done in 32.67s
[2025-12-12 15:35:04,996] INFO:   - wrote MiC_TYPE_3_2_LK2L_MiC_TYPE_3_2_LK2L_57129635_3OJkD8bnr7ZBGamr9ZFQ2E.ifc in 0.42s (size=3.47 MB)
[2025-12-12 15:35:04,997] INFO: [2/49] Extracting IfcBuildingElementProxy | GlobalId=3OJkD8bnr7ZBGamr9ZFQ6h | Name='MiC_TYPE 3.1_LK1L:MiC_TYPE 3.1_LK1L:57129862'
[2025-12-12 15:35:24,321] INFO:   - ifcpatch ExtractElements done in 19.32s
[2025-12-12 15:35:24,589] INFO:   - wrote MiC_TYPE_3_1_LK1L_MiC_TYPE_3_1_LK1L_57129862_3OJkD8bnr7ZBGamr9ZFQ6h.ifc in 0.27s (size=2.45 MB)
[2025-12-12 15:35:24,589] INFO: [3/49] Extracting IfcBuildingElementProxy | GlobalId=3OJkD8bnr7ZBGamr9ZFQO0 | Name='MiC_TYPE 3_LK1L:MiC_TYPE 3_LK1L:5713002

'mic_individuals\\export_index.csv'

In [ ]:
ifcopenshell.__version__

'0.8.4'